In [2]:
# Application Description:
# FPGA-based hardware accelerator for ResMLP model.
# *** Software/Hardware Co-simulation for testing purposes ***

# Instructions:
# 1. Ensure you have the libraries installed (pip install timm torch pillow)
# 2. Run the jupyter notebook cells in order
# 3. Feel free to change the model name to any timm ResMLP model
# 4. Feel free to change the input image url to any image of your choice

# Performance:
# This simulates 12 consecutive layers of the ResMLP architecture. 
# Token-Mixing is calculated in Float32 (Software).
# Channel-Mixing is calculated using Int16/Int8 Bit-Accurate Emulation (Hardware).

In [3]:
"""
Load the pre-trained ResMLP model and Hardware Emulators
"""

from urllib.request import urlopen
from PIL import Image
import timm
import torch
import torch.nn.functional as F

# --- Bit-Accurate Hardware Emulation Functions ---
ACT = 16
FRC = 12

def float_to_fixed_int16(tensor, frac_bits=12):
    scale = 2 ** frac_bits
    return torch.round(tensor * scale).clamp(-32768, 32767).to(torch.int32)

def float_to_fixed_int8(tensor, frac_bits=4):
    scale = 2 ** frac_bits
    return torch.round(tensor * scale).clamp(-128, 127).to(torch.int32)

def hw_emul(a, b): return ((torch.bitwise_right_shift(a * b, 12) + 32768) % 65536 - 32768)
def hw_eadd(a, b): return ((a + b + 32768) % 65536 - 32768)
def hw_outgb(a):   return torch.clamp(torch.bitwise_right_shift(a + 128, 8), -128, 127).to(torch.int8)
def hw_promote(a): return torch.bitwise_left_shift(a.to(torch.int32), 8).to(torch.int32)

def hw_matmul(a_int8, w_int8):
    accum = torch.matmul(a_int8.to(torch.int32), w_int8.T.to(torch.int32))
    return torch.clamp(torch.bitwise_right_shift(accum * 167, 11), -32768, 32767).to(torch.int16)

def hw_gelu(a_int16):
    g_float = F.gelu(a_int16.to(torch.float32) / (2 ** FRC))
    return torch.round(g_float * (2 ** FRC)).clamp(-32768, 32767).to(torch.int32)

model_name = 'resmlp_12_224.fb_in1k'

model = timm.create_model(model_name, pretrained=True)
model = model.eval()

stem = model.stem
blocks = model.blocks    # List of ResMLP layers
norm = model.norm        # Final affine norm layer
head = model.head        # Classification head (Linear)

# Get model-specific transforms (resize to 224x224, normalization, etc.)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

print(f"Loaded model: {model_name}")
print(f"Num blocks: {len(blocks)}")
print(f"Hidden dim (num_features): {model.num_features}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model: resmlp_12_224.fb_in1k
Num blocks: 12
Hidden dim (num_features): 384


In [4]:
"""
Software Step: Process image(s) for input
"""

# Load and convert image
url = 'https://hips.hearstapps.com/hmg-prod/images/schnauzer-carrying-lead-6643258113665.jpeg'
image = Image.open(urlopen(url)).convert("RGB")
input_tensor = transforms(image).unsqueeze(0)  # [1, 3, 224, 224]
print(f"Input tensor shape: {input_tensor.shape}")

# Patch embedding for input tokens
with torch.no_grad():
    x = stem(input_tensor)
    print(f"Shape after patch embedding: {x.shape}")
    # Output: [B, num_patches, hidden_dim] = [1, 196, 384]

Input tensor shape: torch.Size([1, 3, 224, 224])
Shape after patch embedding: torch.Size([1, 196, 384])


In [5]:
"""
Hardware Step: ResMLP Blocks
Each block has two sub-layers:
    (a) Token-mixing: Computed in Float32 Software (CPU)
    (b) Channel-mixing: Computed in Int16/Int8 Bit-Accurate Emulation (FPGA)
"""
with torch.no_grad():
    for i, block in enumerate(blocks):
        print(f"Processing Block {i+1}/12...")

        # =====================================================================
        # (a) Token-Mixing Sublayer (SOFTWARE / CPU)
        # =====================================================================
        
        # 1. Affine pre-norm
        norm1_alpha = block.norm1.alpha.detach()
        norm1_beta  = block.norm1.beta.detach() if hasattr(block.norm1, 'beta') else 0
        x_scaled_1  = (x * norm1_alpha) + norm1_beta

        # Transpose so Linear operates across the token (spatial) dimension
        x_scaled_1_T = x_scaled_1.transpose(1, 2)  # [B, hidden_dim, num_patches]

        # 2. Token-mixing Linear weights & biases
        tok_w = block.linear_tokens.weight.detach()  # [196, 196]
        tok_b = block.linear_tokens.bias.detach()

        # Single linear projection
        tok_out_T = torch.matmul(x_scaled_1_T, tok_w.T) + tok_b
        tok_out = tok_out_T.transpose(1, 2)          # [B, num_patches, hidden_dim]

        # =====================================================================
        # (b) Channel-Mixing MLP Sublayer (HARDWARE EMULATION / FPGA)
        # =====================================================================
        
        # CPU to FPGA Handoff (Quantization)
        hw_x       = float_to_fixed_int16(x)
        hw_tok_out = float_to_fixed_int16(tok_out)
        
        hw_ls1     = float_to_fixed_int16(block.ls1.detach())
        hw_n2_a    = float_to_fixed_int16(block.norm2.alpha.detach())
        hw_n2_b    = float_to_fixed_int16(block.norm2.beta.detach() if hasattr(block.norm2, 'beta') else torch.zeros_like(block.norm2.alpha))
        hw_fc1_b   = float_to_fixed_int16(block.mlp_channels.fc1.bias.detach())
        hw_fc2_b   = float_to_fixed_int16(block.mlp_channels.fc2.bias.detach())
        hw_ls2     = float_to_fixed_int16(block.ls2.detach())
        
        hw_fc1_w   = float_to_fixed_int8(block.mlp_channels.fc1.weight.detach())
        hw_fc2_w   = float_to_fixed_int8(block.mlp_channels.fc2.weight.detach())

        # Phase 1: LS1 + Res -> x_tok (8-bit output)
        ph1_emul = hw_emul(hw_tok_out, hw_ls1)
        hw_x_tok = hw_outgb(hw_eadd(ph1_emul, hw_x))

        # Phase 2: Norm 2 -> x_scaled_2 (8-bit output)
        x_tok_promoted = hw_promote(hw_x_tok)
        ph2_emul = hw_emul(x_tok_promoted, hw_n2_a)
        hw_x_scl = hw_outgb(hw_eadd(ph2_emul, hw_n2_b))

        # Phase 3: MatMul 1 -> ch_hidden_raw (16-bit accumulator)
        hw_raw1 = hw_matmul(hw_x_scl, hw_fc1_w)

        # Phase 4: Bias + GELU -> ch_hidden (8-bit output)
        ph4_eadd = hw_eadd(hw_raw1, hw_fc1_b)
        hw_ch_hid = hw_outgb(hw_gelu(ph4_eadd))

        # Phase 5: MatMul 2 -> ch_out_raw (16-bit accumulator)
        hw_raw2 = hw_matmul(hw_ch_hid, hw_fc2_w)

        # Phase 6: Bias 2 + LS2 + Res -> final_out (8-bit output)
        ph6_eadd1 = hw_eadd(hw_raw2, hw_fc2_b)
        ph6_emul  = hw_emul(ph6_eadd1, hw_ls2)
        hw_final  = hw_outgb(hw_eadd(ph6_emul, x_tok_promoted))

        # FPGA to CPU Handoff (Dequantization)
        x = hw_final.to(torch.float32) / (2**4)

Processing Block 1/12...
Processing Block 2/12...
Processing Block 3/12...
Processing Block 4/12...
Processing Block 5/12...
Processing Block 6/12...
Processing Block 7/12...
Processing Block 8/12...
Processing Block 9/12...
Processing Block 10/12...
Processing Block 11/12...
Processing Block 12/12...


In [6]:
"""
Software Step: Final feature embedding
"""

with torch.no_grad():
    
    # Final layer norm
    # Equivalent to model.forward_features(transforms(image).unsqueeze(0))
    x = norm(x) # [B, num_patches, hidden_dim]

    # Global average pool over 196 tokens
    # Equivalent to model.forward_head(x, pre_logits=True)
    feature_embedding = x.mean(dim=1) # [B, hidden_dim]
    print(f"Shape of feature embedding: {feature_embedding.shape}")

Shape of feature embedding: torch.Size([1, 384])


In [7]:
"""
Classification Verification
"""

ENABLE_CLASSIFICATION = True

if ENABLE_CLASSIFICATION:
    with torch.no_grad():
        # [1, 1000]
        logits = head(feature_embedding)
        probs = torch.nn.functional.softmax(logits[0], dim=0)
        top5 = torch.topk(probs, 5)

    from timm.data.imagenet_info import ImageNetInfo
    # 1000 classes
    imagenet_info = ImageNetInfo()
    label_descriptions = imagenet_info.label_descriptions(detailed=True, as_dict=False)

    print("\n=== TOP 5 PREDICTIONS (HARDWARE EMULATED) ===")
    for prob, idx in zip(top5.values, top5.indices):
        pred_label = label_descriptions[idx.item()]
        print(f"{prob.item():.4f} - {pred_label} (ID: {idx.item()})")


=== TOP 5 PREDICTIONS (HARDWARE EMULATED) ===
0.0057 - crib, cot: baby bed with high sides made of slats (ID: 520)
0.0047 - Petri dish: a shallow dish used to culture bacteria (ID: 712)
0.0043 - nematode, nematode worm, roundworm: unsegmented worms with elongated rounded body pointed at both ends; mostly free-living but some are parasitic (ID: 111)
0.0042 - airship, dirigible: a steerable self-propelled aircraft (ID: 405)
0.0038 - shower curtain: a curtain that keeps water from splashing out of the shower area (ID: 794)
